In [1]:
!pip install google-play-scraper langdetect pandas

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     -------------------- ----------------- 524.3/981.5 kB 8.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786.4/981.5 kB 2.5 MB/s eta 0:00:01
     ------------------------------ ------- 786


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from google_play_scraper import reviews, Sort

apps = {
    "Netflix": "com.netflix.mediaclient",
    "Disney+": "com.disney.disneyplus",
    "Viu": "com.vuclip.viu",
    "HBO Max": "com.wbd.stream",
    "Prime Video": "com.amazon.avod.thirdpartyclient"
}

TARGET_PER_APP = 600
all_reviews = []

for app_name, app_id in apps.items():
    collected = 0
    token = None
    while collected < TARGET_PER_APP:
        result, token = reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=200,
            continuation_token=token
        )
        if not result:
            break
        for r in result:
            r['app_name'] = app_name
        all_reviews.extend(result)
        collected += len(result)
        if token is None:
            break
    print(f"{app_name}: {collected} reviews collected")

df_raw = pd.DataFrame(all_reviews)
print("Total raw data:", len(df_raw))
df_raw.head()

Netflix: 600 reviews collected
Disney+: 600 reviews collected
Viu: 600 reviews collected
HBO Max: 600 reviews collected
Prime Video: 600 reviews collected
Total raw data: 3000


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,e6415f5c-2bff-40b3-b58f-0c8d899a147c,Steven Ainsworth (Steve-O),https://play-lh.googleusercontent.com/a-/ALV-U...,"I like Netflix, what I don't like is how to ca...",4,0,8.71.0 build 7 50428,2026-10-29 11:24:27,NaN,NaT,8.71.0 build 7 50428,Netflix
1,2e1eb4d3-e396-414f-b589-2a420d00d1a0,Indrajeet Dutta,https://play-lh.googleusercontent.com/a-/ALV-U...,"Excellent, but relising of hindi movie is very...",5,0,NaN,2026-10-29 11:17:36,NaN,NaT,NaN,Netflix
2,f7280824-9140-405b-a77a-82022dbc7a2f,Lady Ghost,https://play-lh.googleusercontent.com/a-/ALV-U...,Out of this world. Thanks Netflix.,5,0,9.90.0 build 10 74712,2026-10-29 11:17:17,NaN,NaT,9.90.0 build 10 74712,Netflix
3,34ea9800-d6c3-4d73-846d-f2bd9a143f17,atreyu alexander,https://play-lh.googleusercontent.com/a-/ALV-U...,it's really good but I w8sh they had more stuf...,4,0,9.87.1 build 9 74620,2026-10-29 11:10:48,NaN,NaT,9.87.1 build 9 74620,Netflix
4,85d9d5f9-fd4c-44ff-96d4-a74ed29300fb,Rachael A Tulk,https://play-lh.googleusercontent.com/a-/ALV-U...,love this show!,5,0,NaN,2026-10-29 11:06:03,NaN,NaT,NaN,Netflix


In [3]:
from langdetect import detect

df = df_raw[['app_name', 'content', 'score', 'at']].copy()

df = df.dropna(subset=['content'])
df = df[df['content'].str.strip() != '']
df = df.drop_duplicates(subset=['content'])

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

df = df[df['content'].apply(is_english)]
print("Total after filter:", len(df))
print(df['score'].value_counts().sort_index())

Total after filter: 2103
score
1    883
2    192
3    146
4    156
5    726
Name: count, dtype: int64


In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 2103 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   app_name  2103 non-null   str           
 1   content   2103 non-null   str           
 2   score     2103 non-null   int64         
 3   at        2103 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 351.0 KB


In [6]:
df_final = df.sample(n=1000, random_state=42).reset_index(drop=True)

print("Total data:", len(df_final))
print("\nRating Distribution:")
print(df_final['score'].value_counts().sort_index())
print("\nApplication Distribution:")
print(df_final['app_name'].value_counts())

Total data: 1000

Rating Distribution:
score
1    409
2     93
3     69
4     79
5    350
Name: count, dtype: int64

Application Distribution:
app_name
Disney+        247
HBO Max        212
Prime Video    197
Netflix        189
Viu            155
Name: count, dtype: int64


In [8]:
df_final.to_csv("../data/reviews_film.csv", index=False)